In [15]:
# ============================================================
# CREDIT CARD FRAUD DETECTION
# Dataset: Simulated transactions Jan 2019 - Dec 2020
# 1000 customers | 800 merchants
# Target: is_fraud (1 = Fraud, 0 = Legitimate)
# ============================================================

# --- Core ---
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# --- Visualization ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px

# --- Preprocessing ---
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# --- Imbalance Handling ---
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# --- Models ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# --- Evaluation ---
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)


In [21]:
test = pd.read_csv("fraudTest.csv")
display(train)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
555714,555714,2020-12-31 23:59:07,30560609640617,fraud_Reilly and Sons,health_fitness,43.77,Michael,Olson,M,558 Michael Estates,...,40.4931,-91.8912,519,Town planner,1966-02-13,9b1f753c79894c9f4b71f04581835ada,1388534347,39.946837,-91.333331,0
555715,555715,2020-12-31 23:59:09,3556613125071656,fraud_Hoppe-Parisian,kids_pets,111.84,Jose,Vasquez,M,572 Davis Mountains,...,29.0393,-95.4401,28739,Futures trader,1999-12-27,2090647dac2c89a1d86c514c427f5b91,1388534349,29.661049,-96.186633,0
555716,555716,2020-12-31 23:59:15,6011724471098086,fraud_Rau-Robel,kids_pets,86.88,Ann,Lawson,F,144 Evans Islands Apt. 683,...,46.1966,-118.9017,3684,Musician,1981-11-29,6c5b7c8add471975aa0fec023b2e8408,1388534355,46.658340,-119.715054,0
555717,555717,2020-12-31 23:59:24,4079773899158,fraud_Breitenberg LLC,travel,7.99,Eric,Preston,M,7020 Doyle Stream Apt. 951,...,44.6255,-116.4493,129,Cartographer,1965-12-15,14392d723bb7737606b2700ac791b7aa,1388534364,44.470525,-117.080888,0


In [29]:
train = pd.read_csv("fraudTrain.csv")
train.columns.tolist()

['Unnamed: 0',
 'trans_date_trans_time',
 'cc_num',
 'merchant',
 'category',
 'amt',
 'first',
 'last',
 'gender',
 'street',
 'city',
 'state',
 'zip',
 'lat',
 'long',
 'city_pop',
 'job',
 'dob',
 'trans_num',
 'unix_time',
 'merch_lat',
 'merch_long',
 'is_fraud']

In [31]:
train.describe() # to get quck summary like coumt, mean, std, min, and max

,Unnamed: 0,cc_num,amt,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,is_fraud
count,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06
mean,6.483370e+05,4.171920e+17,7.035104e+01,4.880067e+04,3.853762e+01,-9.022634e+01,8.882444e+04,1.349244e+09,3.853734e+01,-9.022646e+01,5.788652e-03
std,3.743180e+05,1.308806e+18,1.603160e+02,2.689322e+04,5.075808e+00,1.375908e+01,3.019564e+05,1.284128e+07,5.109788e+00,1.377109e+01,7.586269e-02
min,0.000000e+00,6.041621e+10,1.000000e+00,1.257000e+03,2.002710e+01,-1.656723e+02,2.300000e+01,1.325376e+09,1.902779e+01,-1.666712e+02,0.000000e+00
25%,3.241685e+05,1.800429e+14,9.650000e+00,2.623700e+04,3.462050e+01,-9.679800e+01,7.430000e+02,1.338751e+09,3.473357e+01,-9.689728e+01,0.000000e+00
50%,6.483370e+05,3.521417e+15,4.752000e+01,4.817400e+04,3.935430e+01,-8.747690e+01,2.456000e+03,1.349250e+09,3.936568e+01,-8.743839e+01,0.000000e+00
75%,9.725055e+05,4.642255e+15,8.314000e+01,7.204200e+04,4.194040e+01,-8.015800e+01,2.032800e+04,1.359385e+09,4.195716e+01,-8.023680e+01,0.000000e+00
max,1.296674e+06,4.992346e+18,2.894890e+04,9.978300e+04,6.669330e+01,-6.795030e+01,2.906700e+06,1.371817e+09,6.751027e+01,-6.695090e+01,1.000000e+00


In [45]:
train["amt"] 

train[["amt", "category", "is_fraud"]]

fraud_only = train[train["is_fraud"] == 1]
legit_only = train[train["is_fraud"] == 0]

print(f"Fraud transactions: {len(fraud_only):,}")
print(f"Legit transactions: {len(legit_only):,}")

high_fraud = train[(train["is_fraud"] == 1) & (train["amt"]>500)]
print(f"Fraud transactions: {len(high_fraud):,}")


Fraud transactions: 7,506
Legit transactions: 1,289,169
Fraud transactions: 3,648


In [61]:
# for aggregations
train["category"].value_counts() # will count of each vaulue in a column

train.groupby("category")["is_fraud"].mean().sort_values(ascending=False)
train.groupby("category")["amt"].mean().sort_values(ascending=False)
train.groupby("category")["amt"].mean().agg(["mean", "min", "max", "count"])

mean      70.770274
min       47.967678
max      116.960986
count     14.000000
Name: amt, dtype: float64

In [87]:
fraud_rate = (len(train[train["is_fraud"] == 1])) / len(train["is_fraud"]) 
print("The fraud rate is: ", fraud_rate*100)

The fraud rate is:  0.5788651743883394


In [85]:
train.sort_values("amt", ascending=False).head(10)
train[train["is_fraud"] == 1]["merchant"].value_counts().head(10)
train[train["amt"]>1000].head(10)

train["state"].unique()
train["category"].nunique() # how many unique categories

14

In [95]:
high_fraud_state = train[train["is_fraud"] == 1]["state"].value_counts().head(1)
print("The state that has the most amount of fraud is: ", high_fraud_state)

The state that has the most amount of fraud is:  state
NY    555
Name: count, dtype: int64


## Data Cleaning

In [102]:
print(train.isnull().sum())
print(test.isnull().sum())

Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64
Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat            

In [108]:
print(train.duplicated().sum())
print(test.duplicated().sum())

0
0


In [112]:
train.dtypes
test.dtypes

Unnamed: 0                 int64
trans_date_trans_time     object
cc_num                     int64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
dtype: object

In [118]:
# trans_date_trans_time is a string — convert to datetime
train["trans_date_trans_time"] = pd.to_datetime(train["trans_date_trans_time"])
# dob is also a string — convert to datetime
train["dob"] = pd.to_datetime(train["dob"])
train[["trans_date_trans_time", "dob"]].dtypes


trans_date_trans_time    datetime64[ns]
dob                      datetime64[ns]
dtype: object

In [ ]:
#removing columns we do not need

cols_to_drop = ["trans_num", "Unnamed: 0"]

cols_to_drop_train = [c for c in cols_to_drop if c in train.columns]
cols_to_drop_test = [c for c in cols_to_drop if c in test.columns]

train = train.drop(columns=cols_to_drop)
test = te.drop(columns=cols_to_drop)

